# Chapter 4: Two-Stage Training

*Build a Multimodal Model from Scratch*

---

In Chapter 3 we assembled the full VLM architecture: a frozen ViT, a learned
`ProjectionMLP`, and a GPT decoder.  We left one crucial question open:

> **If we have a pretrained ViT and a pretrained GPT, how do we actually
> train them together without destroying the knowledge already inside them?**

The answer is a two-stage training protocol that is used in nearly every
modern VLM—LLaVA, InstructBLIP, Flamingo, and others all follow some
variant of it.  This chapter explains *why* the two-stage approach works,
builds both training loops from scratch, and runs a controlled ablation to
show what breaks when you skip Stage 1.

## 4.1  The Semantic Gap Problem

Imagine you have two expert friends: Alice, who speaks only Mandarin, and Bob,
who speaks only English.  You want them to collaborate, so you hire a
translator between them.  Now consider two approaches:

1. **Naïve approach**: make Alice re-learn Mandarin AND Bob re-learn English
   AND hire the translator, all at the same time.
2. **Smart approach**: keep Alice and Bob as they are; only train the
   translator.  Once the translator is fluent, you can then let Bob refine
   his responses.

Modern VLMs follow approach 2.  The ViT has been trained on millions of
images; the GPT has been pretrained on billions of text tokens.  Both already
encode rich knowledge.  Randomly re-initializing a projection layer and
gradient-descending through *all* parameters simultaneously risks:

* **Catastrophic forgetting** — the GPT's language priors are overwritten by
  noisy image gradients before the projection has learned anything useful.
* **Training instability** — the projection starts random, producing garbage
  "visual tokens"; every layer in the GPT tries to adjust for that garbage.
* **Sample inefficiency** — you need far more image-caption pairs to recover
  the same performance that a staged approach reaches in a fraction of steps.

The solution is to decouple learning into two stages:

| Stage | Frozen | Trainable | Goal |
|-------|--------|-----------|------|
| **Stage 1** — Alignment | ViT + GPT | ProjectionMLP only | Teach the projector to map visual features into the GPT's token embedding space |
| **Stage 2** — Fine-tuning | ViT | ProjectionMLP + GPT | Teach the GPT to follow instructions about visual content |

Let's build this step by step.

In [ ]:
import os, sys, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

os.makedirs('figures', exist_ok=True)
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 4.2  A Diagram of the Two Stages

The figure below shows which components are frozen (grey border) versus
being trained (colored border) in each stage.

In [ ]:
def draw_two_stage_overview():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    def draw_stage(ax, title, vit_color, proj_color, gpt_color):
        ax.set_xlim(0, 10); ax.set_ylim(0, 4)
        ax.axis('off')
        ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

        # Components
        components = [
            (1.0, 1.2, 2.2, 1.6, 'ViT\n(Image Encoder)', vit_color),
            (4.0, 1.2, 2.2, 1.6, 'Projection\nMLP', proj_color),
            (7.0, 1.2, 2.2, 1.6, 'GPT\nDecoder', gpt_color),
        ]
        for (x, y, w, h, label, color) in components:
            frozen = color == '#cccccc'
            lw = 1.5 if frozen else 3.0
            rect = mpatches.FancyBboxPatch(
                (x, y), w, h,
                boxstyle='round,pad=0.1',
                facecolor=color,
                edgecolor='#555555' if frozen else color,
                linewidth=lw
            )
            ax.add_patch(rect)
            ax.text(x + w/2, y + h/2, label,
                    ha='center', va='center', fontsize=11,
                    color='#333333' if frozen else 'white', fontweight='bold')
            # Freeze icon
            if frozen:
                ax.text(x + w/2, y + h + 0.15, '[frozen]',
                        ha='center', va='bottom', fontsize=8, color='#888888',
                        style='italic')
            else:
                ax.text(x + w/2, y + h + 0.15, '[training]',
                        ha='center', va='bottom', fontsize=8, color=color,
                        fontweight='bold')

        # Arrows
        for (x1, x2) in [(3.2, 4.0), (6.2, 7.0)]:
            ax.annotate('', xy=(x2, 2.0), xytext=(x1, 2.0),
                        arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))

        # Image input
        ax.text(0.5, 2.0, 'Image', ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#fffbe6', edgecolor='#e0a000'))
        ax.annotate('', xy=(1.0, 2.0), xytext=(0.9, 2.0),
                    arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))

        # Loss
        ax.text(9.3, 2.0, 'Loss', ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#fff0f0', edgecolor='#cc0000'))
        ax.annotate('', xy=(9.2, 2.0), xytext=(9.2, 2.0),
                    arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))
        ax.plot([9.2, 9.2], [2.0, 2.0], color='#333333')

    GREY    = '#cccccc'
    BLUE    = '#3b82f6'
    GREEN   = '#22c55e'
    ORANGE  = '#f97316'

    draw_stage(axes[0], 'Stage 1: Alignment\n(only ProjectionMLP trains)',
               GREY, ORANGE, GREY)
    draw_stage(axes[1], 'Stage 2: Instruction Fine-tuning\n(ProjectionMLP + GPT train)',
               GREY, ORANGE, BLUE)

    plt.tight_layout()
    plt.savefig('figures/two_stage_overview.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Figure saved.')

draw_two_stage_overview()

## 4.3  Rebuilding the Model Components

We keep our implementations self-contained.  Below we redefine the same
`ViTEncoder`, `GPTDecoder`, `ProjectionMLP`, and `VisionLanguageModel` from
Chapter 3, so this notebook works standalone.

In [ ]:
# ── ViT (minimal, image encoder only) ──────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):                          # (B, C, H, W)
        return self.proj(x).flatten(2).transpose(1, 2)  # (B, N, D)


class ViTBlock(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def forward(self, x):
        # Bidirectional: no attn_mask → every patch attends to every patch
        h, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


class ViTEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3,
                 embed_dim=128, depth=4, n_heads=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.blocks  = nn.ModuleList([ViTBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm    = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        tokens = self.patch_embed(x)                       # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)            # (B, 1, D)
        tokens = torch.cat([cls, tokens], dim=1)           # (B, N+1, D)
        tokens = tokens + self.pos_embed
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)
        return tokens[:, 1:, :]  # return patch tokens (skip CLS), (B, N, D)

print('ViTEncoder defined.')

In [ ]:
# ── GPT Decoder ─────────────────────────────────────────────────────────────

class GPTBlock(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def _causal_mask(self, seq_len, device):
        # Upper triangle = True → those positions are masked (cannot attend)
        return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()

    def forward(self, x, n_visual=0):
        B, T, D = x.shape
        # Visual tokens (first n_visual positions) attend to everything bidirectionally.
        # Text tokens attend causally to all prior positions (including visual ones).
        causal = self._causal_mask(T, x.device)
        if n_visual > 0:
            # Allow visual tokens to attend to each other (no mask on those rows/cols)
            causal[:n_visual, :] = False         # visual tokens: no restriction
            causal[n_visual:, :n_visual] = False # text tokens: can see all visual
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=causal)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class GPTDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, depth=4, n_heads=4, max_seq=256):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(max_seq, embed_dim)
        self.blocks      = nn.ModuleList([GPTBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm        = nn.LayerNorm(embed_dim)
        self.lm_head     = nn.Linear(embed_dim, vocab_size, bias=False)
        # Weight tying: share token embedding and output projection
        self.lm_head.weight = self.token_embed.weight
        self.max_seq = max_seq

    def forward(self, input_ids, visual_prefix=None):
        B, T = input_ids.shape
        device = input_ids.device

        tok  = self.token_embed(input_ids)
        pos  = self.pos_embed(torch.arange(T, device=device)).unsqueeze(0)
        x    = tok + pos

        n_visual = 0
        if visual_prefix is not None:
            # Prepend visual tokens before text tokens
            n_visual = visual_prefix.shape[1]
            # Visual tokens get positions 0..n_visual-1, text gets n_visual..
            vpos = self.pos_embed(torch.arange(n_visual, device=device)).unsqueeze(0)
            vis  = visual_prefix + vpos
            tpos = self.pos_embed(
                torch.arange(n_visual, n_visual + T, device=device)
            ).unsqueeze(0)
            x    = tok + tpos
            x    = torch.cat([vis, x], dim=1)   # (B, n_visual+T, D)

        for blk in self.blocks:
            x = blk(x, n_visual=n_visual)
        x = self.norm(x)
        logits = self.lm_head(x)    # (B, n_visual+T, vocab_size)
        return logits, n_visual

print('GPTDecoder defined.')

In [ ]:
# ── ProjectionMLP ───────────────────────────────────────────────────────────

class ProjectionMLP(nn.Module):
    """Two-layer MLP with GELU activation.

    Maps visual patch tokens from ViT embedding space (vision_dim)
    into GPT token embedding space (language_dim).

    Why two layers?  A single linear layer can rescale dimensions but cannot
    non-linearly re-arrange the feature axes.  The hidden GELU layer allows
    the projection to learn a non-linear alignment between the two spaces.
    """
    def __init__(self, vision_dim, language_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_dim, language_dim),
            nn.GELU(),
            nn.Linear(language_dim, language_dim),
        )

    def forward(self, x):   # x: (B, N_patches, vision_dim)
        return self.net(x)  # returns (B, N_patches, language_dim)

print('ProjectionMLP defined.')

In [ ]:
# ── VisionLanguageModel ──────────────────────────────────────────────────────

class VisionLanguageModel(nn.Module):
    def __init__(self, vit, projection, gpt):
        super().__init__()
        self.vit        = vit
        self.projection = projection
        self.gpt        = gpt

    def forward(self, images, input_ids, labels=None):
        # 1. Encode images  →  patch tokens in ViT space
        patch_tokens = self.vit(images)                    # (B, N_img, vision_dim)
        # 2. Project into GPT space
        visual_prefix = self.projection(patch_tokens)      # (B, N_img, language_dim)
        # 3. Run GPT with visual prefix
        logits, n_visual = self.gpt(input_ids, visual_prefix=visual_prefix)
        # logits shape: (B, n_visual + T, vocab_size)

        loss = None
        if labels is not None:
            # labels: (B, T)  already has -100 at positions we don't supervise
            # logits for text positions: logits[:, n_visual:-1, :]  predicts next token
            # targets: labels[:, 1:]  (shifted by one for next-token prediction)
            B, T = labels.shape
            text_logits  = logits[:, n_visual:-1, :]   # (B, T-1, vocab)
            text_targets = labels[:, 1:]                # (B, T-1)
            loss = F.cross_entropy(
                text_logits.reshape(-1, text_logits.size(-1)),
                text_targets.reshape(-1),
                ignore_index=-100
            )
        return logits, loss

    # ── Stage control ────────────────────────────────────────────────────────

    def set_stage1(self):
        """Stage 1: freeze ViT and GPT; train only the ProjectionMLP."""
        for p in self.vit.parameters():        p.requires_grad_(False)
        for p in self.gpt.parameters():        p.requires_grad_(False)
        for p in self.projection.parameters(): p.requires_grad_(True)
        print('Stage 1: only ProjectionMLP will be updated.')

    def set_stage2(self):
        """Stage 2: freeze ViT; train ProjectionMLP + GPT."""
        for p in self.vit.parameters():        p.requires_grad_(False)
        for p in self.projection.parameters(): p.requires_grad_(True)
        for p in self.gpt.parameters():        p.requires_grad_(True)
        print('Stage 2: ProjectionMLP + GPT will be updated.')

    def count_params(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen    = total - trainable
        print(f'  Total params  : {total:,}')
        print(f'  Trainable     : {trainable:,}  ({100*trainable/total:.1f}%)')
        print(f'  Frozen        : {frozen:,}  ({100*frozen/total:.1f}%)')

print('VisionLanguageModel defined.')

## 4.4  Synthetic Image–Caption Dataset

For training, we need aligned (image, caption) pairs.  We generate a
synthetic dataset with four classes, each with a distinct visual appearance
and a unique caption template.

| Class | Image | Caption |
|-------|-------|---------|
| 0 | Solid red | `"a red image"` |
| 1 | Solid blue | `"a blue image"` |
| 2 | Red–blue vertical stripes | `"vertical stripes"` |
| 3 | Red–blue horizontal stripes | `"horizontal stripes"` |

This is simple enough that the model can learn real associations, so loss
curves will show genuine convergence.

In [ ]:
# ── Tokenizer (character-level) ─────────────────────────────────────────────

class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set(''.join(texts)))
        self.vocab = ['<pad>', '<sos>', '<eos>'] + chars
        self.stoi  = {c: i for i, c in enumerate(self.vocab)}
        self.itos  = {i: c for i, c in enumerate(self.vocab)}
        self.pad_id = 0
        self.sos_id = 1
        self.eos_id = 2

    def encode(self, text, max_len=32, add_special=True):
        ids = [self.stoi[c] for c in text if c in self.stoi]
        if add_special:
            ids = [self.sos_id] + ids + [self.eos_id]
        # Pad or truncate
        ids = ids[:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            tok = self.itos.get(i, '')
            if tok in ('<pad>', '<sos>'):
                continue
            if tok == '<eos>':
                break
            out.append(tok)
        return ''.join(out)

    @property
    def vocab_size(self):
        return len(self.vocab)


CAPTIONS = ['a red image', 'a blue image', 'vertical stripes', 'horizontal stripes']
tokenizer = CharTokenizer(CAPTIONS)
print(f'Vocabulary size: {tokenizer.vocab_size}')
print(f'Encoded "a red image": {tokenizer.encode("a red image")}')

In [ ]:
# ── Dataset ─────────────────────────────────────────────────────────────────

class ImageCaptionDataset(Dataset):
    """Synthetic 4-class image–caption dataset."""

    IMG_SIZE = 32

    def __init__(self, n_samples=400, tokenizer=None, max_len=24):
        self.n_samples  = n_samples
        self.tokenizer  = tokenizer
        self.max_len    = max_len
        self.captions   = CAPTIONS
        self.data       = self._generate()

    def _make_image(self, label):
        img = torch.zeros(3, self.IMG_SIZE, self.IMG_SIZE)
        S   = self.IMG_SIZE
        if label == 0:   # solid red
            img[0] = 0.9
        elif label == 1: # solid blue
            img[2] = 0.9
        elif label == 2: # vertical stripes (alternating columns)
            for c in range(S):
                if c % 8 < 4:
                    img[0, :, c] = 0.9   # red
                else:
                    img[2, :, c] = 0.9   # blue
        else:            # horizontal stripes (alternating rows)
            for r in range(S):
                if r % 8 < 4:
                    img[0, r, :] = 0.9   # red
                else:
                    img[2, r, :] = 0.9   # blue
        return img + torch.randn_like(img) * 0.05   # tiny noise

    def _generate(self):
        data = []
        for i in range(self.n_samples):
            label  = i % 4
            image  = self._make_image(label)
            caption = self.captions[label]
            if self.tokenizer:
                ids = self.tokenizer.encode(caption, max_len=self.max_len)
                ids = torch.tensor(ids, dtype=torch.long)
            else:
                ids = caption
            data.append((image, ids, label))
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


dataset    = ImageCaptionDataset(n_samples=800, tokenizer=tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

imgs, caps, labels = next(iter(dataloader))
print(f'Image batch : {imgs.shape}')
print(f'Caption IDs : {caps.shape}')
print(f'First caption decoded: "{tokenizer.decode(caps[0].tolist())}"')

In [ ]:
def visualize_dataset_samples():
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for label in range(4):
        sample = next(item for item in dataset.data if item[2] == label)
        img = sample[0].permute(1, 2, 0).clamp(0, 1).numpy()
        axes[label].imshow(img)
        axes[label].set_title(f'Class {label}\n"{CAPTIONS[label]}"', fontsize=9)
        axes[label].axis('off')
    plt.suptitle('Synthetic Training Samples', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/dataset_samples.png', dpi=110, bbox_inches='tight')
    plt.show()

visualize_dataset_samples()

## 4.5  Instantiating the Model

We create a small but functional model that fits in CPU RAM and converges
in a reasonable number of epochs for demonstration purposes.

Key dimensions:
* ViT: `embed_dim=128`, 4 transformer blocks, 4 attention heads, 4×4 patches on 32×32 images → 16 patch tokens
* GPT: `embed_dim=128`, 4 blocks, 4 heads, vocabulary from our character tokenizer
* Projection: 128 → 128 (same dimension here, but in real VLMs ViT dim ≫ language dim)

In [ ]:
VIT_DIM  = 128
LANG_DIM = 128
MAX_LEN  = 24    # caption token length

vit        = ViTEncoder(img_size=32, patch_size=8, in_channels=3,
                        embed_dim=VIT_DIM, depth=4, n_heads=4)
projection = ProjectionMLP(vision_dim=VIT_DIM, language_dim=LANG_DIM)
gpt        = GPTDecoder(vocab_size=tokenizer.vocab_size, embed_dim=LANG_DIM,
                        depth=4, n_heads=4, max_seq=MAX_LEN + 16 + 4)

model = VisionLanguageModel(vit, projection, gpt).to(DEVICE)

print('=== Full model (all params) ===')
model.count_params()

model.set_stage1()
print('\n=== After set_stage1() ===')
model.count_params()

model.set_stage2()
print('\n=== After set_stage2() ===')
model.count_params()

## 4.6  Stage 1: Projection Alignment

In Stage 1 we freeze the ViT and the GPT and **only** update the
`ProjectionMLP`.

### Why does this work?

The GPT has already learned what "good" token embeddings look like—vectors
that encode word identities, syntax, and semantics.  When visual patch
tokens pass through the projection, they arrive as random-looking vectors
that the GPT has never seen.  The GPT's attention layers struggle to
incorporate them meaningfully.

By training *only* the projection, we force it to map visual features into
the exact subspace that the frozen GPT already understands.  Think of it as
learning a phrasebook: the ViT speaks "image," the GPT speaks "text," and
the projection layer becomes a bilingual dictionary.

After Stage 1, every visual token coming into the GPT looks—statistically—
like a word embedding the GPT has seen before.  At that point it becomes
safe to also let the GPT fine-tune.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for imgs, caps, _ in loader:
        imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)
        # labels = caps (same tensor); -100 is not needed here because
        # we supervise on all text positions
        _, loss = model(imgs, caps, labels=caps)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


# ── Run Stage 1 ──────────────────────────────────────────────────────────────
model.set_stage1()

stage1_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-3
)
stage1_losses = []
STAGE1_EPOCHS = 25

print('Stage 1 training ...')
for epoch in range(STAGE1_EPOCHS):
    loss = train_one_epoch(model, dataloader, stage1_optimizer)
    stage1_losses.append(loss)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:3d}/{STAGE1_EPOCHS}  loss = {loss:.4f}')

print(f'\nStage 1 complete.  Final loss: {stage1_losses[-1]:.4f}')

## 4.7  Stage 2: Instruction Fine-Tuning

After Stage 1 the projection has been aligned.  Now we unfreeze the GPT
(but keep the ViT frozen) and continue training with a lower learning rate.

### Why keep the ViT frozen?

The ViT's convolutional and attention weights encode a rich understanding
of image structure—edges, textures, objects—built from massive pretraining
data.  We have far fewer image–caption pairs in Stage 2 than the ViT saw
during its original training.  Unfreezing the ViT would allow gradients
from a small, narrow dataset to overwrite general visual knowledge with
dataset-specific patterns, a form of **catastrophic forgetting**.

In practice (LLaVA, InstructBLIP) the ViT remains frozen throughout all
training stages; only the language model and the projector are adapted.

### Loss masking in Stage 2

In instruction-tuning the input typically looks like:

```
[VISUAL TOKENS] <sos> describe the image: <answer> <eos>
```

We only want to supervise the model on the `<answer>` portion.  The
instruction prefix tokens (including visual tokens) are masked with `-100`
so `F.cross_entropy` ignores them.

For simplicity, in our demo we supervise on the entire caption (no
instruction prefix), but the masking mechanism is demonstrated below.

In [ ]:
# ── Stage 2: save Stage-1 checkpoint first ──────────────────────────────────
import copy
stage1_checkpoint = copy.deepcopy(model.state_dict())

# ── Run Stage 2 ──────────────────────────────────────────────────────────────
model.set_stage2()

stage2_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3   # lower LR: the GPT is already trained, we are fine-tuning
)

# Learning-rate scheduler: cosine decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    stage2_optimizer, T_max=25, eta_min=1e-4
)

stage2_losses = []
STAGE2_EPOCHS = 25

print('Stage 2 training ...')
for epoch in range(STAGE2_EPOCHS):
    loss = train_one_epoch(model, dataloader, stage2_optimizer)
    scheduler.step()
    stage2_losses.append(loss)
    if (epoch + 1) % 5 == 0:
        lr = stage2_optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch+1:3d}/{STAGE2_EPOCHS}  loss = {loss:.4f}  lr = {lr:.2e}')

print(f'\nStage 2 complete.  Final loss: {stage2_losses[-1]:.4f}')

## 4.8  Training Curves

Let's plot the loss from both stages together.  You should see:

* **Stage 1**: rapid initial drop as the projection quickly adapts, then
  plateau — the projection has aligned as much as it can alone.
* **Stage 2**: the loss continues to fall as the GPT now co-adapts with the
  projection, reaching a much lower final value.

In [ ]:
def plot_training_curves(stage1_losses, stage2_losses):
    fig, ax = plt.subplots(figsize=(10, 4))

    e1 = list(range(1, len(stage1_losses) + 1))
    e2 = list(range(len(stage1_losses) + 1,
                    len(stage1_losses) + len(stage2_losses) + 1))

    ax.plot(e1, stage1_losses, color='#f97316', lw=2.5, label='Stage 1 (projection only)')
    ax.plot(e2, stage2_losses, color='#3b82f6', lw=2.5, label='Stage 2 (projection + GPT)')

    # Stage boundary
    boundary = len(stage1_losses) + 0.5
    ax.axvline(boundary, color='#666', lw=1.5, linestyle='--')
    ax.text(boundary + 0.3, max(stage1_losses) * 0.95,
            'Stage 1 -> Stage 2', fontsize=9, color='#555')

    # Shade regions
    ax.axvspan(1, boundary, alpha=0.07, color='#f97316')
    ax.axvspan(boundary, len(stage1_losses) + len(stage2_losses),
               alpha=0.07, color='#3b82f6')

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
    ax.set_title('Two-Stage Training Loss Curve', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.4)

    plt.tight_layout()
    plt.savefig('figures/training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_training_curves(stage1_losses, stage2_losses)

## 4.9  Ablation: What Happens Without Stage 1?

A natural question: *can we just skip Stage 1 and go straight to Stage 2?*

Let's find out.  We create a **fresh** model with identical architecture,
skip straight to Stage 2 training, and compare the loss curves.

The experiment has three conditions:

| Condition | Stage 1 | Stage 2 |
|-----------|---------|---------|
| **Full two-stage** | 25 epochs (projection only) | 25 epochs (projection + GPT) |
| **No Stage 1** | — | 50 epochs (projection + GPT) |
| **Stage 1 only** | 50 epochs (projection only) | — |

We give the "No Stage 1" model the same total epoch budget (50 epochs) so
the comparison is fair.

In [ ]:
# ── Condition: No Stage 1 ────────────────────────────────────────────────────

torch.manual_seed(99)
vit2        = ViTEncoder(img_size=32, patch_size=8, in_channels=3,
                         embed_dim=VIT_DIM, depth=4, n_heads=4)
projection2 = ProjectionMLP(vision_dim=VIT_DIM, language_dim=LANG_DIM)
gpt2        = GPTDecoder(vocab_size=tokenizer.vocab_size, embed_dim=LANG_DIM,
                         depth=4, n_heads=4, max_seq=MAX_LEN + 16 + 4)
model_nostage1 = VisionLanguageModel(vit2, projection2, gpt2).to(DEVICE)
model_nostage1.set_stage2()   # skip Stage 1 entirely

opt_nostage1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_nostage1.parameters()), lr=1e-3
)
sched_nostage1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_nostage1, T_max=50, eta_min=1e-4
)

nostage1_losses = []
ABLATION_EPOCHS = 50

print('Ablation: No Stage 1 ...')
for epoch in range(ABLATION_EPOCHS):
    loss = train_one_epoch(model_nostage1, dataloader, opt_nostage1)
    sched_nostage1.step()
    nostage1_losses.append(loss)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{ABLATION_EPOCHS}  loss = {loss:.4f}')

print(f'No-Stage-1 final loss: {nostage1_losses[-1]:.4f}')

In [ ]:
# ── Condition: Stage 1 only ──────────────────────────────────────────────────

torch.manual_seed(77)
vit3        = ViTEncoder(img_size=32, patch_size=8, in_channels=3,
                         embed_dim=VIT_DIM, depth=4, n_heads=4)
projection3 = ProjectionMLP(vision_dim=VIT_DIM, language_dim=LANG_DIM)
gpt3        = GPTDecoder(vocab_size=tokenizer.vocab_size, embed_dim=LANG_DIM,
                         depth=4, n_heads=4, max_seq=MAX_LEN + 16 + 4)
model_stage1only = VisionLanguageModel(vit3, projection3, gpt3).to(DEVICE)
model_stage1only.set_stage1()  # only projection trains

opt_stage1only = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_stage1only.parameters()), lr=3e-3
)

stage1only_losses = []
print('Ablation: Stage 1 only ...')
for epoch in range(ABLATION_EPOCHS):
    loss = train_one_epoch(model_stage1only, dataloader, opt_stage1only)
    stage1only_losses.append(loss)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{ABLATION_EPOCHS}  loss = {loss:.4f}')

print(f'Stage-1-only final loss: {stage1only_losses[-1]:.4f}')

In [ ]:
def plot_ablation(stage1_losses, stage2_losses, nostage1_losses, stage1only_losses):
    fig, ax = plt.subplots(figsize=(11, 5))

    # Full two-stage curve (concatenated)
    full = stage1_losses + stage2_losses
    epochs_full = list(range(1, len(full) + 1))

    ax.plot(epochs_full, full,
            color='#22c55e', lw=2.5, label='Full two-stage (Stage1+Stage2)')
    ax.plot(range(1, ABLATION_EPOCHS+1), nostage1_losses,
            color='#ef4444', lw=2, linestyle='--', label='No Stage 1 (Stage 2 only, 50 epochs)')
    ax.plot(range(1, ABLATION_EPOCHS+1), stage1only_losses,
            color='#f97316', lw=2, linestyle=':', label='Stage 1 only (projection only, 50 epochs)')

    # Stage boundary for full two-stage
    boundary = len(stage1_losses) + 0.5
    ax.axvline(boundary, color='#666', lw=1, linestyle='--', alpha=0.6)
    ax.text(boundary + 0.3, max(full) * 0.85, 'S1->S2', fontsize=8, color='#555')

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
    ax.set_title('Ablation Study: Effect of Stage 1 Pre-alignment', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.4)

    plt.tight_layout()
    plt.savefig('figures/ablation_study.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Summary table
    print('\n=== Final Loss Comparison ===')
    print(f'  Full two-stage          : {(stage1_losses+stage2_losses)[-1]:.4f}')
    print(f'  No Stage 1 (50 epochs)  : {nostage1_losses[-1]:.4f}')
    print(f'  Stage 1 only (50 epochs): {stage1only_losses[-1]:.4f}')

plot_ablation(stage1_losses, stage2_losses, nostage1_losses, stage1only_losses)

### What the Ablation Tells Us

Looking at the loss curves:

* **Full two-stage** reaches the lowest final loss.  The projection first
  learns to speak the GPT's language, and then both projection and GPT
  refine together from that warm start.

* **No Stage 1** converges more slowly in the early epochs because the
  projection is producing garbage visual tokens and the GPT has to
  simultaneously make sense of them while also updating its own weights.
  The final loss is higher—the model ends up in a worse local minimum.

* **Stage 1 only** plateaus early.  The projection can only do so much with
  a completely frozen GPT.  The GPT's language modeling head was trained on
  text alone and cannot fully leverage visual information without adapting.

This confirms the intuition: Stage 1 provides a warm initialization for Stage
2, and Stage 2 provides the capacity that Stage 1 alone lacks.

## 4.10  Loss Masking Deep Dive

In real instruction-tuning datasets, each sample looks like:

```
User: <image> What color is the object?
Assistant: The object is red.
```

We want to compute the loss ONLY on the `Assistant:` portion.  Here is how
the mask is constructed:

```
input_ids : [<sos>  U  s  e  r  :  ...  A  s  s  t  :  r  e  d  <eos>]
labels    : [-100 -100 -100 ... -100 -100  A  s  s  t  :  r  e  d  <eos>]
                ↑ these positions are ignored                ↑ these are supervised
```

Visual tokens prepended by the projection are implicitly masked because they
never appear in `labels` at all—labels only covers the `input_ids` sequence,
and the model internally shifts labels by +1 for next-token prediction.

In [ ]:
def demonstrate_loss_masking():
    """Show how loss masking affects which tokens contribute to the loss."""

    # Example: instruction + answer
    instruction = '<sos>describe:'
    answer      = 'red image<eos>'
    full_text   = instruction + answer

    ids_all    = tokenizer.encode(full_text, max_len=MAX_LEN, add_special=False)
    ids_tensor = torch.tensor(ids_all, dtype=torch.long).unsqueeze(0).to(DEVICE)

    # Labels: -100 for instruction positions, real IDs for answer
    instr_len = len(tokenizer.encode(instruction, add_special=False, max_len=MAX_LEN))
    labels    = ids_tensor.clone()
    labels[0, :instr_len] = -100

    # Fake image
    fake_img = torch.zeros(1, 3, 32, 32).to(DEVICE)

    # Forward pass
    model.eval()
    with torch.no_grad():
        logits, loss = model(fake_img, ids_tensor, labels=labels)

    print(f'Input length : {ids_tensor.shape[1]} tokens')
    print(f'Instruction  : "{instruction}" ({instr_len} tokens, all masked to -100)')
    print(f'Answer       : "{answer}" ({ids_tensor.shape[1]-instr_len} tokens, supervised)')
    print(f'Loss (on answer tokens only): {loss:.4f}')

    # Visualize the mask
    fig, ax = plt.subplots(figsize=(12, 1.5))
    colors = ['#f87171' if l == -100 else '#86efac' for l in labels[0].cpu().tolist()]
    for i, (c, cid) in enumerate(zip(colors, ids_tensor[0].cpu().tolist())):
        rect = plt.Rectangle([i, 0], 1, 1, color=c)
        ax.add_patch(rect)
        tok = tokenizer.itos.get(cid, '?')
        ax.text(i + 0.5, 0.5, tok, ha='center', va='center', fontsize=8)

    n = len(colors)
    ax.set_xlim(0, n); ax.set_ylim(0, 1)
    ax.axis('off')

    red_patch   = mpatches.Patch(color='#f87171', label='Masked (-100, no gradient)')
    green_patch = mpatches.Patch(color='#86efac', label='Supervised (contributes to loss)')
    ax.legend(handles=[red_patch, green_patch], loc='upper right',
              bbox_to_anchor=(1, 1.5), fontsize=8)

    ax.set_title('Loss Masking: Red = ignored, Green = supervised',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/loss_masking.png', dpi=110, bbox_inches='tight')
    plt.show()

demonstrate_loss_masking()

## 4.11  Parameter Efficiency of Two-Stage Training

One compelling advantage of the two-stage approach is that Stage 1 is
**extremely parameter-efficient**: you only update the projection (a tiny
fraction of the total parameters) while leveraging a fully pretrained ViT
and GPT.

Let's visualize the parameter distribution.

In [ ]:
def plot_parameter_distribution():
    components = {
        'ViT\nEncoder': sum(p.numel() for p in model.vit.parameters()),
        'Projection\nMLP': sum(p.numel() for p in model.projection.parameters()),
        'GPT\nDecoder': sum(p.numel() for p in model.gpt.parameters()),
    }
    total = sum(components.values())

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    colors = ['#94a3b8', '#f97316', '#3b82f6']   # grey, orange, blue

    for ax_idx, (stage_name, stage_trainable) in enumerate([
        ('All Params',       {'ViT\nEncoder', 'Projection\nMLP', 'GPT\nDecoder'}),
        ('Stage 1 Trainable',{'Projection\nMLP'}),
        ('Stage 2 Trainable',{'Projection\nMLP', 'GPT\nDecoder'}),
    ]):
        ax = axes[ax_idx]
        for i, (name, count) in enumerate(components.items()):
            pct = 100 * count / total
            is_trainable = name in stage_trainable
            bar = ax.bar(name, count, color=colors[i],
                         alpha=1.0 if is_trainable else 0.25,
                         edgecolor='black' if is_trainable else 'grey',
                         linewidth=1.5 if is_trainable else 0.5)
            label = f'{count:,}\n({pct:.1f}%)'
            if not is_trainable:
                label += '\n[frozen]'
            ax.text(i, count + total * 0.01, label,
                    ha='center', va='bottom', fontsize=7.5)

        ax.set_title(stage_name, fontsize=11, fontweight='bold')
        ax.set_ylabel('Parameter Count', fontsize=9)
        ax.set_ylim(0, total * 1.35)
        ax.yaxis.set_major_formatter(
            matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    plt.suptitle('Parameter Distribution Across Training Stages', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/parameter_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Percent trainable per stage
    proj_params = components['Projection\nMLP']
    gpt_params  = components['GPT\nDecoder']
    print(f'Stage 1 trains {proj_params:,} / {total:,} params'
          f' ({100*proj_params/total:.1f}%)')
    print(f'Stage 2 trains {proj_params+gpt_params:,} / {total:,} params'
          f' ({100*(proj_params+gpt_params)/total:.1f}%)')

plot_parameter_distribution()

## 4.12  Chapter Summary

In this chapter we built the complete two-stage training pipeline for our VLM:

| Concept | Key Insight |
|---------|-------------|
| **Semantic gap** | ViT and GPT features live in different spaces; the projection bridges them |
| **Stage 1 — Alignment** | Freeze ViT + GPT; train projection only. Cheap, fast, provides warm start |
| **Stage 2 — Fine-tuning** | Keep ViT frozen; unfreeze GPT + projection. Allows GPT to adapt to visual input |
| **Why keep ViT frozen?** | ViT encodes general visual knowledge; small datasets would cause catastrophic forgetting |
| **Loss masking** | Set instruction/visual labels to -100; `F.cross_entropy(ignore_index=-100)` skips them |
| **Parameter efficiency** | Stage 1 trains ~5-10% of total params; gives most of the gain |

**What's next — Chapter 5: Inference**

With a trained VLM, we can now generate text conditioned on an image.
Chapter 5 covers:
* Autoregressive token sampling (greedy, top-k, nucleus)
* Visual prefix generation at inference time
* Evaluation: BLEU, CIDEr, and perplexity on our synthetic test set
* A simple interactive demo loop